<div align="center">

## UNIVERSITATEA DE STAT DIN MOLDOVA
## FACULTATEA DE MATEMATICĂ ȘI INFORMATICĂ
## DEPARTAMENT INFORMATICĂ




<br>




### Lucrare Individuală Nr. 2. `Analiza statistică avansată a datelor și metode de învățare automată`
### Metode Statistice de Analiză a Datelor




<br>




#### Studentii: Godoroja Oxana, Daniil Socolov, Djurinschi Andr

_____________________

# Анализ влияния сезонности и возобновляемой генерации на цену электроэнергии в Испании

## 1. Введение и обоснование темы

## 1.1 Актуальность темы

Электроэнергетика является критической инфраструктурой любой современной экономики.
Цены на электроэнергию зависят от множества факторов: структуры генерации,
погодных, рыночных условий и сезонного спроса. Умение предсказывать
цены и потребление позволяет операторам энергосистем оптимизировать загрузку
мощностей, снижать затраты и планировать переход на возобновляемые источники энергии (ВИЭ).

Испания — один из крупнейших производителей солнечной и ветровой энергии в Европе,
что делает её энергосистему особенно интересным объектом для анализа влияния ВИЭ
на ценообразование. Данный анализ позволяет выявить закономерности, применимые
при построении систем краткосрочного прогнозирования цен на электроэнергию.


## 1.2 Цели анализа

1. **Разведочный анализ (EDA):** Выявить закономерности в распределении цен,
   нагрузки и генерации; проанализировать сезонные и почасовые паттерны.

2. **Регрессия:** Построить модели предсказания фактической цены (`price_actual`)
   на основе показателей генерации и погодных условий.

3. **Классификация:** Разделить ценовые уровни на категории (Low / High) и
   обучить классификатор для их определения.

4. **PCA:** Снизить размерность пространства признаков, выявить латентные факторы,
   объясняющие структуру данных.

5. **SARIMA:** Построить модель временного ряда для `price_actual` и оценить
   качество краткосрочного прогноза.

**Главный исследовательский вопрос:**
> *Как сезонность и доля возобновляемой генерации влияют на рыночную цену электроэнергии в Испании?*

### 1.3 Описание набора данных

В работе используются два взаимосвязанных набора данных:

#### `energy_dataset.csv`
| Параметр | Значение |
|---|---|
| Источник | ENTSO-E Transparency Platform / Kaggle |
| Период | 01.01.2015 — 31.12.2018 |
| Частота | Почасовая |
| Количество наблюдений | ~35 064 строк |
| Ключевые переменные | Генерация по источникам (МВт), прогнозы нагрузки, `total load actual`, `price actual` (€/МВт·ч) |

#### Примеры данных `energy_dataset.csv`:
| time                      | city     | temp    | temp_min | temp_max | pressure | humidity | wind_speed | clouds_all | rain_1h | rain_3h | snow_3h | snow_1h | weather_id | weather_main | weather_description | weather_icon |
|---------------------------|----------|---------|----------|----------|----------|----------|------------|------------|---------|---------|---------|---------|------------|--------------|---------------------|--------------|
| 2015-01-01 00:00:00+01:00 | Valencia | 270.475 | 270.475  | 270.475  | 1001     | 77       | 1          | 62         | 0.0     | 0.0     | 0.0     | 0       | 800        | clear        | sky is clear        | 01n          |
| 2015-01-01 01:00:00+01:00 | Valencia | 270.475 | 270.475  | 270.475  | 1001     | 77       | 1          | 62         | 0.0     | 0.0     | 0.0     | 0       | 800        | clear        | sky is clear        | 01n          |
| 2015-01-01 02:00:00+01:00 | Valencia | 269.686 | 269.686  | 269.686  | 1002     | 78       | 0          | 23         | 0.0     | 0.0     | 0.0     | 0       | 800        | clear        | sky is clear        | 01n          |

#### `weather_features.csv`
| Параметр | Значение |
|---|---|
| Источник | OpenWeatherMap API / Kaggle |
| Период | 01.01.2015 — 31.12.2018 |
| Частота | Почасовая |
| Города | Valencia, Madrid, Bilbao, Barcelona, Seville |
| Ключевые переменные | Температура (К), влажность, давление, скорость ветра, облачность, тип погоды |

#### Примеры данных `weather_features.csv`:

| dt_iso                    | city_name | temp    | temp_min | temp_max | pressure | humidity | wind_speed | wind_deg | rain_1h | rain_3h | snow_3h | clouds_all | weather_id | weather_main | weather_description | weather_icon |
|---------------------------|-----------|---------|----------|----------|----------|----------|------------|----------|---------|---------|---------|------------|------------|--------------|---------------------|--------------|
| 2015-01-01 00:00:00+01:00 | Valencia  | 270.475 | 270.475  | 270.475  | 1001     | 77       | 1          | 62       | 0.0     | 0.0     | 0.0     | 0          | 800        | clear        | sky is clear        | 01n          |
| 2015-01-01 01:00:00+01:00 | Valencia  | 270.475 | 270.475  | 270.475  | 1001     | 77       | 1          | 62       | 0.0     | 0.0     | 0.0     | 0          | 800        | clear        | sky is clear        | 01n          |
| 2015-01-01 02:00:00+01:00 | Valencia  | 269.686 | 269.686  | 269.686  | 1002     | 78       | 0          | 23       | 0.0     | 0.0     | 0.0     | 0          | 800        | clear        | sky is clear        | 01n          |

### 1.3.1 Описание переменных — `energy_dataset.csv`

| Переменная | Тип | Описание |
|---|---|---|
| `time` | datetime | Временная метка (UTC+1), почасовая |
| `generation biomass` | float | Генерация из биомассы (МВт) |
| `generation fossil brown coal/lignite` | float | Генерация из бурого угля / лигнита (МВт) |
| `generation fossil coal-derived gas` | float | Генерация из газа на основе угля (МВт) |
| `generation fossil gas` | float | Генерация из природного газа (МВт) |
| `generation fossil hard coal` | float | Генерация из каменного угля (МВт) |
| `generation fossil oil` | float | Генерация из нефти (МВт) |
| `generation fossil oil shale` | float | Генерация из горючих сланцев (МВт) |
| `generation fossil peat` | float | Генерация из торфа (МВт) |
| `generation geothermal` | float | Геотермальная генерация (МВт) |
| `generation hydro pumped storage aggregated` | float | Гидроаккумулирующие станции — агрегированно (МВт) |
| `generation hydro pumped storage consumption` | float | Потребление ГАЭС при закачке воды (МВт) |
| `generation hydro run-of-river and poundage` | float | Русловые ГЭС (МВт) |
| `generation hydro water reservoir` | float | Водохранилищные ГЭС (МВт) |
| `generation marine` | float | Морская (приливная) генерация (МВт) |
| `generation nuclear` | float | Атомная генерация (МВт) |
| `generation other` | float | Прочие источники генерации (МВт) |
| `generation other renewable` | float | Прочие возобновляемые источники (МВт) |
| `generation solar` | float | Солнечная генерация (МВт) |
| `generation waste` | float | Генерация из отходов (МВт) |
| `generation wind offshore` | float | Морская ветровая генерация (МВт) |
| `generation wind onshore` | float | Наземная ветровая генерация (МВт) |
| `forecast solar day ahead` | float | Прогноз солнечной генерации на следующий день (МВт) |
| `forecast wind offshore eday ahead` | float | Прогноз морской ветровой генерации на следующий день (МВт) |
| `forecast wind onshore day ahead` | float | Прогноз наземной ветровой генерации на следующий день (МВт) |
| `total load forecast` | float | Прогнозируемая суммарная нагрузка (МВт) |
| `total load actual` | float | Фактическая суммарная нагрузка (МВт) |
| `price day ahead` | float | Прогнозная цена на следующий день (€/МВт·ч) |
| `price actual` | float | **Целевая переменная** — фактическая рыночная цена (€/МВт·ч) |

---

### 1.3.2 Описание переменных — `weather_features.csv`

| Переменная | Тип | Описание |
|---|---|---|
| `dt_iso` | datetime | Временная метка (UTC+1), почасовая |
| `city_name` | string | Название города (Valencia, Madrid, Bilbao, Barcelona, Seville) |
| `temp` | float | Температура воздуха (Кельвины) |
| `temp_min` | float | Минимальная температура за период (Кельвины) |
| `temp_max` | float | Максимальная температура за период (Кельвины) |
| `pressure` | int | Атмосферное давление (гПа) |
| `humidity` | int | Относительная влажность воздуха (%) |
| `wind_speed` | float | Скорость ветра (м/с) |
| `wind_deg` | int | Направление ветра (градусы, 0–360°) |
| `rain_1h` | float | Количество осадков за последний час (мм) |
| `rain_3h` | float | Количество осадков за последние 3 часа (мм) |
| `snow_3h` | float | Количество снега за последние 3 часа (мм) |
| `clouds_all` | int | Облачность (%, 0 = ясно, 100 = сплошная облачность) |
| `weather_id` | int | Код погодного условия (OpenWeatherMap) |
| `weather_main` | string | Основная категория погоды (Clear, Clouds, Rain, Snow и др.) |
| `weather_description` | string | Детальное описание погоды (на английском) |
| `weather_icon` | string | Код иконки погоды (OpenWeatherMap) |

### 1.3.3 Типы переменных

| Переменная | Тип | Описание |
|---|---|---|
| `time` / `dtiso` | datetime | Временная метка (UTC+1) |
| `price_actual` | float, непрерывная | Фактическая цена на электроэнергию (€/MWh) |
| `total_load_actual` | float, непрерывная | Фактическое потребление (MWh) |
| `generation_*` | float, непрерывная | Выработка по типам источников (MWh) |
| `temp`, `tempmin`, `tempmax` | float, непрерывная | Температура (K) |
| `pressure`, `humidity` | int, непрерывная | Давление (hPa), влажность (%) |
| `windspeed`, `winddeg` | float/int, непрерывная | Скорость и направление ветра |
| `rain1h`, `snow3h` и др. | float, непрерывная | Осадки (мм) |
| `weathermain` | string → int, категориальная | Тип погоды (Clear, Rain, Snow...) |
| `cityname` | string, категориальная | Город (Valencia, Madrid и др.) |
| `season` | string → int, категориальная | Сезон (Winter, Spring, Summer, Autumn) |
| `hour`, `month`, `year` | int, дискретная | Производные временные признаки |
| `renewableshare` | float, непрерывная | Доля ВИЭ в общей генерации (%) |

![img](msadlab2-spain.png)



### 1.4 Вклад членов команды

| Студент | Роль |
|---|---|
| Студент 1 | Определение темы, загрузка и очистка данных, EDA и визуализации |
| Студент 2 | Модели регрессии и классификации, оценка производительности |
| Студент 3 | PCA, анализ временных рядов (ARIMA), выводы и презентация |

## 2. Загрузка и очистка данных

В данном разделе выполняется загрузка исходных данных, их первичный осмотр,
обработка пропущенных значений, выявление и устранение выбросов,
а также подготовка признаков для дальнейшего анализа.

In [ ]:
import pandas as pd
import numpy as np
import plotly.io as pio
import nbformat
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

# Цвета для сезонов
season_colors = {
    'Winter': '#4A90D9',
    'Spring': '#7DBF5A',
    'Summer': '#F5A623',
    'Autumn': '#D0743C'
}

print("✅ Библиотеки загружены")

In [ ]:
energy = pd.read_csv('energy_dataset.csv', parse_dates=['time'])
weather = pd.read_csv('weather_features.csv', parse_dates=['dt_iso'])

print("=== energy_dataset ===")
print(f"Строк: {len(energy)}, Колонок: {len(energy.columns)}")
display(energy.head(3))

print("\n=== weather_features ===")
print(f"Строк: {len(weather)}, Колонок: {len(weather.columns)}")
print(f"Города: {weather['city_name'].unique()}")
display(weather.head(3))

### 2.1 Анализ пропущенных значений

In [ ]:
# Считаем пропуски
missing_energy = (energy.isnull().sum() / len(energy) * 100).round(2)
missing_energy = missing_energy[missing_energy > 0].sort_values(ascending=False).reset_index()
missing_energy.columns = ['Переменная', 'Пропуски (%)']

missing_weather = (weather.isnull().sum() / len(weather) * 100).round(2)
missing_weather = missing_weather[missing_weather > 0].sort_values(ascending=False).reset_index()
missing_weather.columns = ['Переменная', 'Пропуски (%)']

# График для energy
fig1 = px.bar(
    missing_energy,
    x='Переменная', y='Пропуски (%)',
    title='Пропущенные значения — energy_dataset (%)',
    color='Пропуски (%)',
    color_continuous_scale='Blues',
    text='Пропуски (%)'
)
fig1.update_layout(
    template='plotly_white',
    title_font_size=16,
    xaxis_tickangle=-45,
    coloraxis_showscale=False
)
fig1.update_traces(textposition='outside')
fig1.show()

# График для weather
if len(missing_weather) > 0:
    fig2 = px.bar(
        missing_weather,
        x='Переменная', y='Пропуски (%)',
        title='Пропущенные значения — weather_features (%)',
        color='Пропуски (%)',
        color_continuous_scale='Greens',
        text='Пропуски (%)'
    )
    fig2.update_layout(
        template='plotly_white',
        title_font_size=16,
        xaxis_tickangle=-45,
        coloraxis_showscale=False
    )
    fig2.update_traces(textposition='outside')
    fig2.show()
else:
    print("✅ В weather_features пропусков нет")

### Интерпретация пропущенных значений

**energy_dataset:**
- Колонки `generation fossil oil shale`, `generation fossil peat`, `generation marine`,
  `generation wind offshore` и `generation hydro pumped storage aggregated` содержат
  значительную долю пропусков — это объясняется тем, что данные источники генерации
  либо отсутствуют в Испании, либо не отслеживались в отдельные периоды.
- Колонки с >50% пропусков будут удалены как неинформативные.
- Остальные пропуски заполняются медианным значением, что устойчиво к выбросам.

**weather_features:**
- Переменные `rain_1h`, `rain_3h`, `snow_3h` содержат пропуски, поскольку
  осадки — редкое явление: отсутствие значения фактически означает 0 мм.
  Пропуски заполняются нулём.
- Прочие метеорологические переменные практически полны.

### Обоснование решений по пропускам

- **Столбцы с >50% пропусков удалены** (`generation fossil oil shale`,
  `generation fossil peat`, `generation marine`, `generation wind offshore`,
  `generation hydro pumped storage aggregated`, `forecast wind offshore eday ahead`):
  при таком количестве пропусков любое заполнение искажает реальную картину.
  Оставлять их бессмысленно — модели будут обучаться на "придуманных" данных.

- **Остальные числовые пропуски заполнены медианой**, а не средним:
  медиана устойчива к выбросам. В данных об энергетике бывают экстремальные
  значения (например, пиковая нагрузка в жару), которые сильно смещают среднее.

- **Пропуски в `rain1h`, `rain3h`, `snow3h` заполнены нулями:**
  отсутствие записи об осадках в большинстве случаев означает их отсутствие —
  это физически обоснованное решение.

### 2.2 Выявление и обработка выбросов

Выбросы могут искажать результаты моделирования и статистические оценки.
Для выявления используется метод межквартильного размаха (IQR).
Визуально выбросы отображаются с помощью box plots для ключевых переменных.

In [ ]:
key_vars = ['price actual', 'total load actual', 'generation solar', 'generation wind onshore']

fig = make_subplots(rows=1, cols=4, subplot_titles=key_vars)

colors = ['#4A90D9', '#7DBF5A', '#F5A623', '#D0743C']

for i, col in enumerate(key_vars):
    fig.add_trace(
        go.Box(
            y=energy[col].dropna(),
            name=col,
            marker_color=colors[i],
            boxmean=True
        ),
        row=1, col=i + 1
    )

fig.update_layout(
    template='plotly_white',
    title_text='Распределение ключевых переменных (до обработки выбросов)',
    title_font_size=16,
    showlegend=False,
    height=500
)
fig.show()

In [ ]:
# Подсчёт изменённых значений после clipping
numeric_cols = energy.select_dtypes(include='number').columns
original = energy.copy()  # если у вас сохранена копия до обработки

clipped_count = 0
for col in numeric_cols:
    Q1 = energy[col].quantile(0.25)
    Q3 = energy[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_clipped = ((energy[col] < lower) | (energy[col] > upper)).sum()
    clipped_count += n_clipped

print(f"Всего значений, изменённых clipping: {clipped_count}")
print(f"Это {clipped_count / (len(energy) * len(numeric_cols)) * 100:.2f}% от всех числовых значений")

### Обоснование обработки выбросов (метод IQR)

Метод IQR (межквартильный размах) — стандартный и устойчивый способ
определения выбросов. Граница выброса:

- Нижняя: Q1 − 1.5 × IQR
- Верхняя: Q3 + 1.5 × IQR

Где **Q1** — 25-й перцентиль, **Q3** — 75-й перцентиль,
**IQR = Q3 − Q1**.

Вместо удаления строк с выбросами мы применили **clipping** (обрезку):
значения за пределами границ заменяются самой граничной величиной.
Это сохраняет количество строк (важно для временных рядов, где удаление
нарушает непрерывность) и одновременно нейтрализует влияние экстремальных значений.

### Интерпретация выбросов

- **`price actual`** — содержит экстремально высокие и низкие (вплоть до отрицательных)
  значения цены. Это **не ошибки измерений**, а реальные рыночные явления:
  - *Высокие цены* — пиковый спрос (зима/лето) при низкой ветровой/солнечной генерации
  - *Низкие и отрицательные цены* — избыток ВИЭ, когда солнце и ветер производят
    больше, чем потребляется, и электроэнергию некуда сохранить
  - Такая волатильность — характерная черта либерализованного энергорынка Испании.
  - **Решение:** выбросы сохраняются, так как несут важную аналитическую информацию.

- **`total load actual`** — экстремальные значения нагрузки связаны с праздничными
  днями или аномальными погодными условиями. Сохраняются как реальные наблюдения.

- **`generation solar` / `generation wind onshore`** — пиковые значения физически
  обоснованы (максимальная генерация в отдельные часы) и обязательно сохраняются
  как ключевые признаки для моделирования.

**Вывод:** в данном датасете метод IQR используется только для **выявления и описания**
аномалий, но не для их удаления. Экстремальные значения отражают реальную динамику
испанского энергорынка и важны для целей прогнозирования.

### 2.3 Подготовка признаков

Создание новых признаков на основе временной метки, расчёт доли возобновляемой
генерации и объединение двух датасетов в единый для дальнейшего анализа.

In [ ]:
# Исправляем парсинг времени
energy['time'] = pd.to_datetime(energy['time'], utc=True).dt.tz_convert('Europe/Madrid').dt.tz_localize(None)

# Временные признаки
energy['hour'] = energy['time'].dt.hour
energy['month'] = energy['time'].dt.month
energy['dayofweek'] = energy['time'].dt.dayofweek
energy['year'] = energy['time'].dt.year


def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'


energy['season'] = energy['month'].apply(get_season)

# Доля возобновляемой генерации
renewable_cols = [c for c in [
    'generation solar', 'generation wind onshore',
    'generation wind offshore', 'generation hydro run-of-river and poundage',
    'generation hydro water reservoir', 'generation biomass',
    'generation other renewable'
] if c in energy.columns]

energy['renewable_share'] = (
        energy[renewable_cols].sum(axis=1) / energy['total load actual'] * 100
)

# Агрегация weather по времени
weather['time'] = pd.to_datetime(weather['dt_iso'], utc=True).dt.tz_convert('Europe/Madrid').dt.tz_localize(None)
weather['time'] = weather['time'].dt.floor('h')

weather_agg = weather.groupby('time').agg({
    'temp': 'mean',
    'humidity': 'mean',
    'wind_speed': 'mean',
    'clouds_all': 'mean',
    'pressure': 'mean',
    'rain_1h': 'mean',
    'weather_main': lambda x: x.mode()[0]
}).reset_index()

# Кодирование weather_main
le = LabelEncoder()
weather_agg['weather_main_enc'] = le.fit_transform(weather_agg['weather_main'])

# Объединение датасетов
df = energy.merge(weather_agg, on='time', how='left')

print(f"✅ Финальный датасет: {df.shape[0]} строк, {df.shape[1]} колонок")
display(df.head(3))

### Итог предобработки

| Шаг | Действие | Результат |
|---|---|---|
| Пропуски >50% | Удаление колонок | Убраны неинформативные признаки |
| Пропуски <50% | Заполнение медианой | Сохранены все наблюдения |
| Выбросы (IQR) | Анализ и сохранение | Экстремальные цены — реальная рыночная динамика |
| Временные признаки | Создание новых колонок | `hour`, `month`, `season`, `year` |
| Доля ВИЭ | Создание `renewable_share` | Ключевой признак для анализа |
| Объединение | Merge по временной метке | Единый датасет для моделирования |

## 3. Анализ данных и визуализации

В данном разделе проводится разведочный анализ данных (EDA) с целью выявления
закономерностей, сезонных паттернов и взаимосвязей между переменными.
Все визуализации интерактивны — наводите курсор для отображения точных значений.

### 3.1 Описательные статистики

In [ ]:
key_cols = ['price actual', 'total load actual', 'renewable_share',
            'generation solar', 'generation wind onshore',
            'temp', 'humidity', 'wind_speed']

stats = df[key_cols].describe().round(2).T
stats.columns = ['Кол-во', 'Среднее', 'Ст. откл.', 'Мин', '25%', 'Медиана', '75%', 'Макс']

display(stats)

### 3.2 Корреляционная матрица

In [ ]:
corr_cols = [
    'price actual', 'total load actual', 'renewable_share',
    'generation solar', 'generation wind onshore', 'generation nuclear',
    'generation fossil gas', 'generation fossil hard coal',
    'temp', 'humidity', 'wind_speed', 'clouds_all', 'pressure'
]

corr = df[corr_cols].corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.columns.tolist(),
    colorscale='RdBu_r',
    zmid=0,
    text=corr.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 9},
    hoverongaps=False
))

fig.update_layout(
    title='Корреляционная матрица ключевых переменных',
    title_font_size=16,
    template='plotly_white',
    height=600,
    xaxis_tickangle=-45
)

fig.show()

### Интерпретация корреляционной матрицы

- **`price actual` и `generation fossil gas`** — положительная корреляция:
  газовые станции часто задают предельную цену на рынке.
- **`price actual` и `renewable_share`** — отрицательная корреляция:
  чем больше доля ВИЭ, тем ниже рыночная цена — ключевой вывод работы.
- **`price actual` и `total load actual`** — положительная:
  высокий спрос толкает цену вверх.
- **`generation solar` и `temp`** — положительная: солнечная генерация
  выше в жаркие часы/месяцы.
- **`generation wind onshore` и `wind_speed`** — положительная:
  очевидная физическая зависимость.


Тепловая карта показывает линейные взаимосвязи между всеми числовыми
переменными. Значение близкое к **+1** означает сильную прямую зависимость,
к **−1** — обратную, **0** — отсутствие линейной связи.

### 3.3 Динамика цены электроэнергии по времени

In [ ]:
df_plot = df[['time', 'price actual', 'season']].dropna().copy()
df_plot = df_plot.sort_values('time')

# Скользящее среднее (7 дней = 168 часов) для сглаживания
df_plot['price_ma'] = df_plot['price actual'].rolling(168, center=True).mean()

season_colors = {
    'Winter': '#4A90D9',
    'Spring': '#7DBF5A',
    'Summer': '#F5A623',
    'Autumn': '#D0743C'
}

fig = go.Figure()

# Фоновая заливка по сезонам
for season, color in season_colors.items():
    mask = df_plot['season'] == season
    fig.add_trace(go.Scatter(
        x=df_plot.loc[mask, 'time'],
        y=df_plot.loc[mask, 'price actual'],
        mode='markers',
        marker=dict(color=color, size=2, opacity=0.3),
        name=season,
        legendgroup=season
    ))

# Скользящее среднее поверх
fig.add_trace(go.Scatter(
    x=df_plot['time'],
    y=df_plot['price_ma'],
    mode='lines',
    line=dict(color='black', width=2),
    name='Скользящее среднее (7 дней)'
))

fig.update_layout(
    title='Динамика цены электроэнергии (2015–2018) по сезонам',
    title_font_size=16,
    template='plotly_white',
    xaxis_title='Дата',
    yaxis_title='Цена (€/МВт·ч)',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

### Интерпретация

- Цена электроэнергии демонстрирует выраженную **сезонную волатильность**:
  пики наблюдаются зимой и летом (высокий спрос на отопление и кондиционирование).
- Весной и осенью цены, как правило, ниже — умеренный спрос совпадает
  с высокой ветровой генерацией.
- Скользящее среднее выявляет общий тренд без краткосрочных шумов.
- В 2018 году наблюдается заметный рост среднего уровня цен по сравнению с 2015–2016 гг.

### 3.4 Тепловая карта цены: час × месяц

Данная визуализация показывает среднюю цену электроэнергии в разрезе
часа суток и месяца года — позволяет одновременно увидеть суточную
и сезонную цикличность.

In [ ]:
pivot = df.groupby(['hour', 'month'])['price actual'].mean().round(2).reset_index()
pivot_matrix = pivot.pivot(index='hour', columns='month', values='price actual')

month_names = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн',
               'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']

fig = go.Figure(data=go.Heatmap(
    z=pivot_matrix.values,
    x=month_names,
    y=[f'{h:02d}:00' for h in pivot_matrix.index],
    colorscale='Plasma',
    colorbar=dict(title='€/МВт·ч'),
    hoverongaps=False,
    hovertemplate='Месяц: %{x}<br>Час: %{y}<br>Цена: %{z:.2f} €/МВт·ч<extra></extra>'
))

fig.update_layout(
    title='Средняя цена электроэнергии по часу суток и месяцу года',
    title_font_size=16,
    template='plotly_white',
    xaxis_title='Месяц',
    yaxis_title='Час суток',
    height=550
)

fig.show()

### Интерпретация

- **Утренний пик (7:00–10:00)** и **вечерний пик (19:00–22:00)** — цена
  стабильно выше в часы максимального потребления.
- **Ночные часы (00:00–05:00)** — цена минимальна во все месяцы.
- **Январь и февраль** в вечерние часы показывают наибольшие значения —
  зимний спрос на отопление накладывается на пик потребления.
- **Летние месяцы (июль–август)** демонстрируют высокие дневные цены
  из-за активного использования кондиционеров.
- Данная тепловая карта наглядно подтверждает двойную цикличность цены:
  **суточную** и **сезонную**.

### 3.5 Структура генерации: возобновляемые vs ископаемые источники

In [ ]:
renewable_cols = [c for c in [
    'generation solar', 'generation wind onshore', 'generation wind offshore',
    'generation hydro run-of-river and poundage', 'generation hydro water reservoir',
    'generation biomass', 'generation other renewable'
] if c in df.columns]

fossil_cols = [c for c in [
    'generation fossil gas', 'generation fossil hard coal',
    'generation fossil brown coal/lignite', 'generation fossil oil',
    'generation fossil coal-derived gas'
] if c in df.columns]

other_cols = [c for c in [
    'generation nuclear', 'generation waste', 'generation other'
] if c in df.columns]

season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
season_labels = ['Зима', 'Весна', 'Лето', 'Осень']

df['fossil_total'] = df[fossil_cols].sum(axis=1)
df['renewable_total'] = df[renewable_cols].sum(axis=1)
df['nuclear_other_total'] = df[other_cols].sum(axis=1)

gen_season = df.groupby('season')[['renewable_total', 'fossil_total', 'nuclear_other_total']].mean().reindex(
    season_order).round(1)

# Считаем проценты
gen_season['total'] = gen_season.sum(axis=1)
gen_season['renewable_pct'] = (gen_season['renewable_total'] / gen_season['total'] * 100).round(1)
gen_season['fossil_pct'] = (gen_season['fossil_total'] / gen_season['total'] * 100).round(1)
gen_season['nuclear_pct'] = (gen_season['nuclear_other_total'] / gen_season['total'] * 100).round(1)

from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Средняя структура генерации по сезонам (МВт)',
        'Доля источников генерации по сезонам (%)'
    )
)

# --- График 1: абсолютные значения (stacked bar) ---
fig.add_trace(go.Bar(
    name='ВИЭ',
    x=season_labels,
    y=gen_season['renewable_total'],
    marker_color='#7DBF5A',
    hovertemplate='%{x}: %{y:.0f} МВт<extra>ВИЭ</extra>'
), row=1, col=1)

fig.add_trace(go.Bar(
    name='Ископаемые',
    x=season_labels,
    y=gen_season['fossil_total'],
    marker_color='#D9534F',
    hovertemplate='%{x}: %{y:.0f} МВт<extra>Ископаемые</extra>'
), row=1, col=1)

fig.add_trace(go.Bar(
    name='Ядерная + прочие',
    x=season_labels,
    y=gen_season['nuclear_other_total'],
    marker_color='#9B59B6',
    hovertemplate='%{x}: %{y:.0f} МВт<extra>Ядерная + прочие</extra>'
), row=1, col=1)

# --- График 2: проценты (stacked bar 100%) ---
fig.add_trace(go.Bar(
    name='ВИЭ',
    x=season_labels,
    y=gen_season['renewable_pct'],
    marker_color='#7DBF5A',
    text=gen_season['renewable_pct'].astype(str) + '%',
    textposition='inside',
    hovertemplate='%{x}: %{y:.1f}%<extra>ВИЭ</extra>',
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Bar(
    name='Ископаемые',
    x=season_labels,
    y=gen_season['fossil_pct'],
    marker_color='#D9534F',
    text=gen_season['fossil_pct'].astype(str) + '%',
    textposition='inside',
    hovertemplate='%{x}: %{y:.1f}%<extra>Ископаемые</extra>',
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Bar(
    name='Ядерная + прочие',
    x=season_labels,
    y=gen_season['nuclear_pct'],
    marker_color='#9B59B6',
    text=gen_season['nuclear_pct'].astype(str) + '%',
    textposition='inside',
    hovertemplate='%{x}: %{y:.1f}%<extra>Ядерная + прочие</extra>',
    showlegend=False
), row=1, col=2)

fig.update_layout(
    barmode='stack',
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1)
)

fig.update_yaxes(title_text='Средняя мощность (МВт)', row=1, col=1)
fig.update_yaxes(title_text='Доля (%)', row=1, col=2)

fig.show()

### Интерпретация

- **Весна** — наибольшая доля ВИЭ (44.5%): активна гидрогенерация
  (таяние снегов) и ветровая генерация. Доля ископаемых минимальна (32.2%) —
  это объясняет более низкие цены весной.
- **Зима** — ВИЭ всё ещё высоки (39.4%), однако ископаемые (37%) подключаются
  для покрытия пикового спроса на отопление, что толкает цену вверх.
- **Лето** — доля ВИЭ падает до 36.6%, несмотря на пик солнечной генерации:
  высокий спрос на кондиционирование требует значительного подключения
  ископаемых источников (39.8%).
- **Осень** — минимальная доля ВИЭ (34.4%) и максимальная доля ископаемых
  (42.4%): ветровая и солнечная генерация снижаются, спрос остаётся умеренным.
- **Ядерная генерация** удивительно стабильна во все сезоны (~23.2–23.6%) —
  подтверждает её роль как базовой нагрузки, независимой от погоды.
- Главный вывод: **чем выше доля ВИЭ — тем ниже давление ископаемых
  на цену**, что подтверждает отрицательную корреляцию из раздела 3.2.

### 3.7 Ветровая генерация и цена электроэнергии
Одна из ключевых гипотез работы — рост ветровой генерации снижает
рыночную цену. График ниже совмещает оба ряда на одной временной оси.

In [ ]:
df_wind = df[['time', 'generation wind onshore', 'price actual']].dropna().copy()
df_wind = df_wind.sort_values('time')

# Недельное скользящее среднее для читаемости
df_wind['wind_ma'] = df_wind['generation wind onshore'].rolling(168, center=True).mean()
df_wind['price_ma'] = df_wind['price actual'].rolling(168, center=True).mean()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=df_wind['time'],
    y=df_wind['wind_ma'],
    mode='lines',
    name='Ветровая генерация (скольз. среднее)',
    line=dict(color='#4A90D9', width=2),
    hovertemplate='%{x}<br>Ветер: %{y:.0f} МВт<extra></extra>'
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_wind['time'],
    y=df_wind['price_ma'],
    mode='lines',
    name='Цена (скольз. среднее)',
    line=dict(color='#F5A623', width=2),
    hovertemplate='%{x}<br>Цена: %{y:.2f} €/МВт·ч<extra></extra>'
), secondary_y=True)

fig.update_layout(
    title='Ветровая генерация vs Цена электроэнергии (2015–2018)',
    title_font_size=16,
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.update_yaxes(title_text='Ветровая генерация (МВт)', secondary_y=False, color='#4A90D9')
fig.update_yaxes(title_text='Цена (€/МВт·ч)', secondary_y=True, color='#F5A623')
fig.update_xaxes(title_text='Дата')

fig.show()

### Интерпретация

- График наглядно демонстрирует **обратную зависимость** между ветровой
  генерацией и ценой электроэнергии: в периоды высокой ветровой активности
  цена заметно снижается.
- Особенно отчётливо эффект виден зимой 2015–2016 гг.: всплески ветровой
  генерации совпадают с падением цены.
- Летом ветровая генерация снижается, а цена растёт — подтверждает
  роль ВИЭ как стабилизатора рынка.
- Данная зависимость является одним из ключевых аргументов в пользу
  расширения ветроэнергетики для сдерживания роста цен.

### 3.8 Распределение цены по типу погоды

In [ ]:
# Оставим только самые частые типы погоды
top_weather = df['weather_main'].value_counts().head(6).index.tolist()
df_weather = df[df['weather_main'].isin(top_weather)].copy()

order = ['Clear', 'Clouds', 'Rain', 'Drizzle', 'Snow', 'Mist']
order = [w for w in order if w in top_weather]

fig = go.Figure()

fig.add_trace(go.Box(
    x=df_weather['weather_main'],
    y=df_weather['price actual'],
    name='Распределение цены',
    boxmean=True,
    marker_color='#4A90D9',
    hovertemplate='Погода: %{x}<br>Цена: %{y:.2f} €/МВт·ч<extra></extra>'
))

fig.update_layout(
    title='Распределение цены электроэнергии по типу погоды',
    title_font_size=16,
    template='plotly_white',
    xaxis_title='Тип погоды (weather_main)',
    yaxis_title='Цена (€/МВт·ч)',
    height=500
)

fig.update_xaxes(categoryorder='array', categoryarray=order)

fig.show()  # Группировка и расчёт статистик (как в boxplot)
stats = df_weather.groupby('weather_main')['price actual'].describe()

# Добавим медиану отдельно (её нет в describe по умолчанию красиво)
stats['median'] = df_weather.groupby('weather_main')['price actual'].median()

# Оставим нужные столбцы
stats = stats[['min', '25%', 'median', '75%', 'max', 'mean']]

# Округлим для красоты
stats = stats.round(2)

print("\n📊 Статистика цен по типу погоды:\n")
print(stats)

### Интерпретация

- **Ясно (clear)** и **облачно (clouds)** — дают очень схожий уровень цен:
  медиана около 58 €/МВт·ч, среднее ~58–59 €/МВт·ч. Погода «без осадков»
  сама по себе не приводит к заметным сдвигам цены.
- При **дожде (rain)** и особенно при **мороси (drizzle)** средняя и медианная
  цена немного ниже (mean ≈ 50–57 €/МВт·ч), что может быть связано
  с более низким спросом в такие периоды и/или усилением ветровой генерации.
- При состояниях **mist/fog** медиана и средняя цена немного выше
  (median ~59–60 €/МВт·ч, mean ~59–60 €/МВт·ч), а верхний квартиль и максимум
  смещены выше: в туманную/смоговую погоду часть ВИЭ (солнечная генерация)
  ограничена, а спрос остаётся стабильным.
- В целом различия между типами погоды **умеренные**: погодные условия
  влияют на цену косвенно — через спрос и доступность ВИЭ, а не напрямую.

In [ ]:
fig = px.violin(df, x="season", y="price actual", box=True, points=False,
                color="season",
                color_discrete_map={"Winter":"#4A90D9","Spring":"#7DBF5A",
                                    "Summer":"#F5A623","Autumn":"#D0743C"},
                category_orders={"season": ["Winter","Spring","Summer","Autumn"]},
                title="Распределение цены на электроэнергию по сезонам",
                labels={"price actual": "Цена (€/MWh)", "season": "Сезон"})
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

In [ ]:
sample = df.sample(5000, random_state=42)
fig = px.scatter(sample, x="temp", y="price actual", color="season",
                 color_discrete_map={"Winter":"#4A90D9","Spring":"#7DBF5A",
                                     "Summer":"#F5A623","Autumn":"#D0743C"},
                 opacity=0.4, trendline="ols",
                 title="Температура vs Цена на электроэнергию",
                 labels={"temp": "Температура (K)", "price actual": "Цена (€/MWh)"})
fig.update_layout(template="plotly_white")
fig.show()

## 4. Модели регрессии и классификации

В этом разделе строятся модели регрессии и классификации для анализа факторов,
влияющих на цену электроэнергии, и для прогнозирования её уровня.
Сначала рассматривается задача регрессии с целевой переменной `price actual`.

В качестве базовой модели регрессии выбран линейный подход (Linear/Ridge/Lasso).

### 4.1 Регрессия: предсказание фактической цены (`price actual`)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler

# Целевая переменная
target = 'price actual'

# Признаки для регрессии
feature_cols = [
    'total load actual',
    'renewable_share',
    'generation fossil gas',
    'generation fossil hard coal',
    'generation nuclear',
    'generation solar',
    'generation wind onshore',
    'temp', 'humidity', 'wind_speed', 'clouds_all', 'pressure',
    'hour', 'dayofweek', 'month',
    'weather_main_enc'
]

feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].dropna()
y = df.loc[X.index, target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f"Признаков: {X.shape[1]}, обучающих объектов: {len(X_train)}, тестовых: {len(X_test)}")

In [ ]:
import numpy as np
import pandas as pd

ml_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
}

results = {}

for name, model in ml_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

results_df = pd.DataFrame(results).T.round(3)
results_df

### Качество моделей регрессии

- **MAE (Mean Absolute Error)** — средняя абсолютная ошибка в €/МВт·ч.
- **RMSE (Root Mean Squared Error)** — сильнее наказывает большие ошибки.
- **R²** — доля объяснённой дисперсии цены (0–1, чем ближе к 1, тем лучше).

**Интерпретация:**

- Средняя абсолютная ошибка около **12 €/МВт·ч**, что сопоставимо
  с типичной волатильностью цены на энергорынке.
- Однако значение **R² ≈ -0.5** указывает, что модель объясняет дисперсию
  хуже, чем наивный константный прогноз (предсказывать всегда среднее значение цены).
- Почти одинаковые метрики у Linear, Ridge и Lasso говорят о том, что
  проблема не в регуляризации, а в самой спецификации модели:
  - неучтён тренд по годам,
  - неучтены нелинейные эффекты и взаимодействия признаков,
  - временная зависимость (автокорреляция) игнорируется.

MAE ≈ 12.1
В среднем модель ошибается на ~12 €/МВт·ч.

RMSE ≈ 14.2
Чуть выше MAE, значит есть довольно крупные ошибки (RMSE сильнее штрафует большие промахи).

R² = -0.498
Это очень важно: отрицательный R² означает, что модель хуже, чем константный прогноз "всегда предсказывать среднее значение цены".

In [ ]:
# Возьмём Ridge как основную модель
best_model = Ridge(alpha=10.0)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

pred_df = pd.DataFrame({
    'time': df.iloc[y_test.index]['time'],
    'y_true': y_test,
    'y_pred': y_pred
}).reset_index(drop=True)


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pred_df['y_true'],
    y=pred_df['y_pred'],
    mode='markers',
    marker=dict(color='#4A90D9', size=4, opacity=0.4),
    name='Наблюдения',
    hovertemplate='Факт: %{x:.2f} €/МВт·ч<br>Прогноз: %{y:.2f} €/МВт·ч<extra></extra>'
))

# Диагональ "идеальный прогноз"
min_val = min(pred_df['y_true'].min(), pred_df['y_pred'].min())
max_val = max(pred_df['y_true'].max(), pred_df['y_pred'].max())

fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Идеальный прогноз'
))

fig.update_layout(
    title='Фактическая vs прогнозируемая цена (Ridge)',
    template='plotly_white',
    xaxis_title='Фактическая цена (€/МВт·ч)',
    yaxis_title='Прогнозируемая цена (€/МВт·ч)',
    height=500
)

fig.show()

### 4.1.2 График «факт vs прогноз»

На диагональной линии расположились бы точки **идеального прогноза**.
Фактические точки заметно разбросаны вокруг диагонали, причём:

- Для низких цен модель часто **завышает** прогноз.
- Для высоких цен — наоборот, **недооценивает**.
- Это типичный эффект линейной модели на сложном, нелинейном и
  высоковолатильном временном ряду.

Такой график визуально подтверждает вывод по метрике R²: текущая
линейная спецификация недостаточно хорошо объясняет поведение цены.

### 4.2 Классификация: уровни цены электроэнергии

Для перехода к задаче классификации непрерывная целевая переменная `price actual`
дискретизируется на три уровня: низкий, средний и высокий. Это позволяет оценить,
насколько хорошо факторы (нагрузка, погода, структура генерации) объясняют переход
между разными «режимами» цен.

In [ ]:
# Используем те же наблюдения, что и для регрессии
data_clf = df.loc[X.index].copy()

# Квантили для порогов
q_low = data_clf['price actual'].quantile(0.33)
q_high = data_clf['price actual'].quantile(0.66)

print(f"Порог низкой цены  (33%): {q_low:.2f} €/МВт·ч")
print(f"Порог высокой цены (66%): {q_high:.2f} €/МВт·ч")

def price_level(p):
    if p <= q_low:
        return 0   # низкая
    elif p <= q_high:
        return 1   # средняя
    else:
        return 2   # высокая

data_clf['price_level'] = data_clf['price actual'].apply(price_level)

data_clf['price_level'].value_counts().sort_index()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X_clf = data_clf[feature_cols].dropna()
y_clf = data_clf.loc[X_clf.index, 'price_level']

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.2, shuffle=False
)

scaler_clf = StandardScaler()
Xc_train_scaled = scaler_clf.fit_transform(Xc_train)
Xc_test_scaled = scaler_clf.transform(Xc_test)

print("Размеры выборок (классификация):")
print("Train:", Xc_train.shape, " Test:", Xc_test.shape)
yc_train.value_counts().sort_index(), yc_test.value_counts().sort_index()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

log_reg = LogisticRegression()  # без доп. параметров

log_reg.fit(Xc_train_scaled, yc_train)
yc_pred = log_reg.predict(Xc_test_scaled)

print("Классификационный отчёт:\n")
print(classification_report(
    yc_test, yc_pred,
    target_names=['Низкая цена', 'Средняя цена', 'Высокая цена']
))

### 4.2.1 Интерпретация результатов классификации


- Целевая переменная `price_level` имеет три уровня:
  - 0 — **низкая цена** (нижние 33% распределения),
  - 1 — **средняя цена** (33–66%),
  - 2 — **высокая цена** (верхние 33%).
- Логистическая регрессия обучена на тех же признаках, что и модель регрессии:
  нагрузка, структура генерации, погода, временные признаки.

По классификационному отчёту:

- **Precision** показывает, насколько «чистым» является каждый предсказанный класс.
- **Recall** — насколько хорошо модель находит все объекты данного класса.
- **F1-score** — баланс между precision и recall.

Матрица ошибок позволяет увидеть, какие классы чаще всего путаются:
обычно модель лучше различает «низкие» и «высокие» цены, а «средний» уровень
часто пересекается с соседними, что логично при построении границ по квантилям.

Классы уровней цены распределены неравномерно: доля наблюдений с высокой
ценой значительно больше, чем с низкой, что отражено в колонке `support`.

По классификационному отчёту:

- Для класса **«Низкая цена»**:
  - precision = 0.25, recall = 0.88, F1 = 0.38.
  - Модель почти все реальные «низкие» цены успешно находит (высокий recall),
    но среди всех предсказанных как «низкие» много ошибок (низкий precision).
    То есть модель **переклассифицирует** часть средних и высоких цен в низкие.

- Для класса **«Средняя цена»**:
  - precision = 0.19, recall = 0.25, F1 = 0.22.
  - Модель хуже всего работает именно с «средним» уровнем: этот класс
    сильно пересекается по признакам с соседними уровнями, и модель
    часто путает его с «низким» или «высоким».

- Для класса **«Высокая цена»**:
  - precision = 0.86, recall = 0.29, F1 = 0.43.
  - Если модель прогнозирует «высокую цену», то в большинстве случаев
    это действительно высокий уровень (высокий precision), однако она
    находит лишь часть всех реально высоких цен (низкий recall).
    То есть модель **осторожно** предсказывает высокие цены и часто
    недооценивает их, относя к среднему или низкому уровню.

- Общая точность (**accuracy ≈ 0.35**) существенно выше случайного угадывания
  при трёх классах (≈ 0.33), но далека от идеала, что ожидаемо для
  шумного и сложного рынка электроэнергии.

- Разрыв между `macro avg` и `weighted avg` показывает влияние
  дисбаланса классов: за счёт большого числа наблюдений с высокой ценой
  взвешенные метрики получаются выше.

В целом, логистическая регрессия способна уловить общие паттерны,
отличающие низкие и высокие уровни цены, но испытывает трудности
с точным разделением «среднего» класса. Это подчёркивает сложность
задачи и ограниченность линейной модели при описании многомерной
и нелинейной структуры данных.

In [ ]:
cm = confusion_matrix(yc_test, yc_pred)

cm_df = pd.DataFrame(
    cm,
    index=['Факт: низкая', 'Факт: средняя', 'Факт: высокая'],
    columns=['Прогноз: низкая', 'Прогноз: средняя', 'Прогноз: высокая']
)

cm_df

In [ ]:
fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=['Прогноз: низкая', 'Прогноз: средняя', 'Прогноз: высокая'],
    y=['Факт: низкая', 'Факт: средняя', 'Факт: высокая'],
    colorscale='Blues',
    text=cm,
    texttemplate='%{text}',
    hovertemplate='%{y}<br>%{x}<br>Количество: %{z}<extra></extra>'
))

fig.update_layout(
    title='Матрица ошибок (логистическая регрессия)',
    template='plotly_white',
    xaxis_title='Прогнозируемый класс',
    yaxis_title='Фактический класс',
    height=500
)

fig.show()

## 5. Снижение размерности: метод главных компонент (PCA)

В этом разделе используется метод главных компонент (Principal Component Analysis, PCA)
для снижения размерности признакового пространства и выявления скрытых структур
в данных об энергосистеме и погоде.

PCA позволяет:
- заменить множество исходных взаимосвязанных признаков меньшим числом
  ортогональных компонент;
- оценить, какая часть общей вариации данных объясняется каждой компонентой;
- увидеть, как объекты (часы наблюдений) группируются в пространстве первых
  компонент (например, по сезонам или уровням цены).

In [ ]:
from sklearn.decomposition import PCA

# Выбираем только числовые признаки для PCA
pca_features = [
    'price actual', 'total load actual', 'renewable_share',
    'generation fossil gas', 'generation fossil hard coal',
    'generation nuclear', 'generation solar', 'generation wind onshore',
    'temp', 'humidity', 'wind_speed', 'clouds_all', 'pressure',
    'hour', 'dayofweek', 'month'
]

pca_features = [c for c in pca_features if c in df.columns]

pca_data = df[pca_features].dropna()
pca_index = pca_data.index

# Стандартизация
from sklearn.preprocessing import StandardScaler
scaler_pca = StandardScaler()
pca_scaled = scaler_pca.fit_transform(pca_data)

print(f"Признаков для PCA: {len(pca_features)}, наблюдений: {pca_scaled.shape[0]}")

In [ ]:
pca = PCA()
pca.fit(pca_scaled)

explained_var = pca.explained_variance_ratio_

pca_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(explained_var))],
    'explained_variance_ratio': explained_var,
    'cumulative_variance': explained_var.cumsum()
})

pca_df.head(10)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=pca_df['PC'],
    y=(pca_df['explained_variance_ratio'] * 100).round(2),
    name='Доля дисперсии, %',
    marker_color='#4A90D9',
    hovertemplate='%{x}: %{y:.2f}%<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=pca_df['PC'],
    y=(pca_df['cumulative_variance'] * 100).round(2),
    mode='lines+markers',
    name='Накопленная доля, %',
    line=dict(color='#F5A623', width=2),
    hovertemplate='%{x}: %{y:.2f}%<extra></extra>'
))

fig.update_layout(
    title='Доля объяснённой дисперсии по главным компонентам',
    template='plotly_white',
    xaxis_title='Главная компонента',
    yaxis_title='Доля дисперсии (%)',
    height=500
)

fig.show()

### 5.1 Интерпретация результатов PCA

По графику доли объяснённой дисперсии видно, что:

- **Первая главная компонента (PC1)** объясняет около **22%** общей вариации признаков.
- **Вторая компонента (PC2)** добавляет ещё примерно **18%**.
- **Третья (PC3)** даёт около **10%**, **четвёртая (PC4)** — ещё ~7%, **пятая (PC5)** — ~6,5%.

Таким образом, первые **5 компонент** суммарно объясняют примерно **64–65%** общей дисперсии данных, а первые **8–10 компонент** покрывают уже подавляющую часть вариации признаков. Начиная с компонент более высокого порядка вклад каждой из них становится существенно меньше (2–4%), что хорошо видно по «колену» на scree plot.

Это означает, что для визуализации структуры данных и построения упрощённых моделей достаточно работать в пространстве первых нескольких компонент (PC1–PC2 или PC1–PC3), практически не теряя ключевую информацию о вариации признаков.

In [ ]:
# Получаем координаты наблюдений в пространстве главных компонент
scores = pca.transform(pca_scaled)
scores_df = pd.DataFrame(scores, columns=[f'PC{i+1}' for i in range(scores.shape[1])], index=pca_index)

# Добавляем к ним сезон и уровень цены
pca_vis = scores_df[['PC1', 'PC2']].copy()
pca_vis['season'] = df.loc[pca_index, 'season'].values
pca_vis['price_level'] = data_clf.loc[pca_index, 'price_level'].values  # 0/1/2

level_map = {0: 'Низкая', 1: 'Средняя', 2: 'Высокая'}
pca_vis['price_level_str'] = pca_vis['price_level'].map(level_map)

In [ ]:
samples = {
    '1 000 точек':  {'n': 1000,                      'size': 6},
    '5 000 точек':  {'n': 5000,                      'size': 4},
    'Все точки':    {'n': len(pca_vis),             'size': 2},
}

fig = go.Figure()

buttons = []
trace_index = 0

for label, cfg in samples.items():
    n = cfg['n']
    size = cfg['size']

    sample_df = pca_vis.sample(n=min(n, len(pca_vis)), random_state=42)

    for season, color in season_colors.items():
        mask = sample_df['season'] == season
        fig.add_trace(go.Scatter(
            x=sample_df.loc[mask, 'PC1'],
            y=sample_df.loc[mask, 'PC2'],
            mode='markers',
            name=f'{season} ({label})',
            marker=dict(color=color, size=size, opacity=0.4),
            visible=(label == '1 000 точек'),  # по умолчанию показываем 1000
            hovertemplate='PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>Сезон: ' + season + '<extra></extra>'
        ))
        trace_index += 1

total_traces = trace_index
traces_per_sample = len(season_colors)

for i, (label, cfg) in enumerate(samples.items()):
    visible = [False] * total_traces
    start = i * traces_per_sample
    end = start + traces_per_sample
    for j in range(start, end):
        visible[j] = True

    buttons.append(dict(
        label=label,
        method='update',
        args=[{'visible': visible},
              {'title': f'PC1–PC2 по сезонам ({label})'}]
    ))

fig.update_layout(
    template='plotly_white',
    xaxis_title='PC1',
    yaxis_title='PC2',
    height=500,
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.5, y=1.15,
        xanchor='center',
        buttons=buttons
    )],
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.update_layout(title='PC1–PC2 по сезонам (1 000 точек)')
fig.show()

### 5.2 Проекция наблюдений на первые главные компоненты

График PC1–PC2 показывает, что:

- Наблюдения разных сезонов занимают **разные области** в пространстве первых двух компонент.
  Это означает, что PCA успешно улавливает сезонную структуру данных
  (нагрузка, генерация, погода) в компактном 2D-представлении.
- Зимние точки (Winter) смещены в одну сторону, летние (Summer) — в другую,
  весна и осень занимают промежуточные зоны. Это согласуется с тем,
  что сезоны отличаются по температурам, структуре генерации и уровню нагрузки.
- Несмотря на пересечения, видны зоны, где преобладают точки одного сезона —
  это свидетельствует о наличии **кластеризации по сезонам** в исходном
  многомерном пространстве признаков.

## 5.3 Biplot: нагрузки признаков на первые две главные компоненты

В этом разделе визуализируем, как исходные признаки загружаются на первые две главные компоненты PCA (PC1 и PC2).
Каждая точка на графике — отдельный признак, а его координаты по осям PC1 и PC2 показывают вклад признака в соответствующую компоненту.
Чем дальше точка от центра, тем сильнее признак влияет на структуру вариации данных по этой компоненте.

In [ ]:
# 5.3 Нагрузки признаков на PC1 и PC2

# Матрица нагрузок (loadings): строки — признаки, столбцы — компоненты
loadings = pca.components_.T[:, :2]  # берём только первые две компоненты

loading_df = pd.DataFrame(
    loadings,
    index=pca_features,          # список исходных признаков, использованных в PCA
    columns=['PC1', 'PC2']
)

print("✅ Нагрузки признаков на первые две главные компоненты")
display(loading_df.round(3))

In [ ]:
# 5.3 Визуализация biplot: нагрузки признаков на PC1 и PC2

fig = go.Figure()

# Точки + подписи для признаков
fig.add_trace(go.Scatter(
    x=loading_df['PC1'],
    y=loading_df['PC2'],
    mode='markers+text',
    text=loading_df.index,
    textposition='top center',
    marker=dict(size=8, color='#D9534F'),
    name='Признаки',
    hovertemplate=(
        'Признак: %{text}<br>'
        'PC1 loading: %{x:.2f}<br>'
        'PC2 loading: %{y:.2f}<extra></extra>'
    )
))

# Горизонтальная и вертикальная оси
fig.add_shape(
    type='line',
    x0=-1, x1=1, y0=0, y1=0,
    line=dict(color='black', width=1)
)
fig.add_shape(
    type='line',
    x0=0, x1=0, y0=-1, y1=1,
    line=dict(color='black', width=1)
)

fig.update_layout(
    title='Biplot: нагрузки исходных признаков на PC1 и PC2',
    template='plotly_white',
    xaxis_title='PC1',
    yaxis_title='PC2',
    xaxis=dict(range=[-1.1, 1.1]),
    yaxis=dict(range=[-1.1, 1.1]),
    height=600
)

fig.show()

### Интерпретация biplot для PC1 и PC2

1. **Первая главная компонента (PC1)**
   - Крупные положительные нагрузки по PC1 у признаков:
     - `generation fossil gas` (0.411),
     - `generation fossil hard coal` (0.397),
     - `price actual` (0.357),
     - `total load actual` (0.325),
     - частично `temp` (0.219), `hour` (0.221), `month` (0.176).
   - В отрицательную сторону по PC1 тянут `renewable_share` (-0.338), `generation wind onshore` (-0.308), `humidity` (-0.188), `clouds_all` (-0.124).
   - Это позволяет интерпретировать PC1 как ось **«дорогой / фоссильный режим vs возобновляемый режим»**:
     - справа (высокий PC1) — высокий спрос, высокая газовая и угольная генерация и более высокая цена;
     - слева (низкий PC1) — более высокая доля ВИЭ и ветра, повышенная влажность/облачность и, как правило, более низкие цены.

2. **Вторая главная компонента (PC2)**
   - Сильные положительные нагрузки по PC2 у:
     - `humidity` (0.455),
     - `generation fossil hard coal` (0.161),
     - `month` (0.131).
   - Сильные отрицательные нагрузки по PC2 у:
     - `renewable_share` (-0.370),
     - `wind_speed` (-0.371),
     - `generation solar` (-0.357),
     - `hour` (-0.336),
     - `temp` (-0.290),
     - `generation wind onshore` (-0.260),
     - `total load actual` (-0.271).
   - PC2 можно трактовать как компоненту **«погодно‑суточного режима ВИЭ»**:
     - в нижней части (отрицательная PC2) — дневные часы с более высокой температурой, скоростью ветра, долей ВИЭ, солнечной и ветровой генерацией и более высокой нагрузкой;
     - в верхней части (положительная PC2) — более влажные и, как правило, менее «солнечно‑ветреные» состояния.

3. **Положение `price actual`**
   - `price actual` имеет заметный положительный loading по PC1 (0.357) и почти нулевой по PC2 (0.063).
   - Это значит, что цена **сильно связана именно с осью PC1** — то есть с уровнем спроса и фоссильной генерацией, а не напрямую с «погодной» осью PC2.
   - Геометрически вектор цены направлен примерно туда же, куда `generation fossil gas` и `total load actual`: при росте спроса и газовой/угольной генерации цена растёт.

4. **Связь цены и доли ВИЭ**
   - `renewable_share` имеет координаты (-0.338, -0.370), то есть уходит в противоположный от `price actual` и фоссильной генерации квадрант.
   - Угол между `price actual` и `renewable_share` близок к 180°, что качественно соответствует **отрицательной зависимости**: когда доля ВИЭ растёт, PC1 и цена уменьшаются.
   - Аналогично `generation wind onshore` (-0.308, -0.260) и `wind_speed` (-0.085, -0.371) направлены против вектора цены — это поддерживает вывод, что ветровая активность и ветроэнергетика сдерживают цену.

5. **Погодные признаки**
   - `temp` (0.219, -0.290) и `hour` (0.221, -0.336) лежат в том же секторе, что и `generation solar`: это отражает дневной/летний режим — тепло, больше солнца, выше солнечная генерация и нагрузка.
   - `humidity` (‑0.188, 0.455) тянет вверх по PC2 и слегка влево по PC1: влажные состояния в среднем связаны с меньшей солнечной генерацией и другой структурой нагрузки.
   - `clouds_all` и `pressure` имеют небольшие по модулю нагрузки — они влияют на компоненты слабее, чем температура, ветер и влажность.

6. **Общий вывод по PCA**
   - Первая компонента PC1 отделяет **дорогий, высоко‑фоссильный режим с высокой нагрузкой** от режима с высокой долей ВИЭ и ветром.
   - Вторая компонента PC2 захватывает **суточно‑погодный паттерн** (день/ночь, тёплые/холодные и влажные/сухие состояния).
   - Цена электроэнергии в этом пространстве однозначно тяготеет к фоссильной стороне PC1 и отдаляется от вектора `renewable_share`, что визуально и количественно подтверждает: рост доли ВИЭ и ветровой генерации связан со снижением цены.

## 6. Анализ временного ряда цены электроэнергии (ARIMA)

В этом разделе рассматриваем цену электроэнергии как временной ряд и моделируем её динамику с помощью ARIMA/SARIMA.
Цель — проверить, насколько хорошо можно предсказывать `price actual` только на основе её прошлых значений, учитывая автокорреляцию и сезонность.

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 6.1 Подготовка временного ряда: дневное среднее price actual

ts = df[['time', 'price actual']].dropna().copy()
ts = ts.set_index('time').sort_index()

# Суточное среднее для сглаживания почасового шума
ts_daily = ts['price actual'].resample('D').mean()

print(f'Количество дневных наблюдений: {len(ts_daily)}')
display(ts_daily.head())

# Визуализация дневной цены
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ts_daily.index,
    y=ts_daily.values,
    mode='lines',
    name='price actual (дневное среднее)',
    line=dict(color='#4A90D9', width=1.5),
    hovertemplate='Дата: %{x}<br>Цена: %{y:.2f} €/МВт·ч<extra></extra>'
))

fig.update_layout(
    title='Дневная динамика цены электроэнергии (price actual)',
    template='plotly_white',
    xaxis_title='Дата',
    yaxis_title='Цена (€/МВт·ч)',
    height=450
)

fig.show()

### 6.1 Визуальный анализ дневного ряда

- Ряд явно нестационарен: присутствует как выраженная сезонность (повторяющиеся годовые и внутригодовые колебания), так и долгосрочный тренд.
- Хорошо видны периоды повышенных цен (зимние и летние пики) и более спокойные интервалы с более низким средним уровнем.
- Для применения ARIMA‑подхода потребуется проверить стационарность и, вероятно, использовать дифференцирование.

In [ ]:
# 6.2 Проверка стационарности (ADF-тест) и дифференцирование

# ADF-тест для исходного дневного ряда
result = adfuller(ts_daily.dropna())
print('ADF-статистика (исходный ряд): {:.3f}'.format(result[0]))
print('p-value: {:.3f}'.format(result[1]))
for key, value in result[4].items():
    print(f'Критическое значение ({key}): {value:.3f}')

# Первое разностное преобразование
ts_diff = ts_daily.diff().dropna()

result_diff = adfuller(ts_diff)
print('\nADF-статистика (после первой разности): {:.3f}'.format(result_diff[0]))
print('p-value: {:.3f}'.format(result_diff[1]))
for key, value in result_diff[4].items():
    print(f'Критическое значение ({key}): {value:.3f}')

### 6.2 Проверка стационарности

- Для исходного дневного ряда `price actual` ADF‑статистика равна -2.798 при p-value = 0.059.
  При уровне значимости 5% критическое значение (-2.864) по модулю чуть больше, чем наблюдаемое, а p-value немного выше 0.05.
  Формально это означает, что на уровне 5% **мы не можем уверенно отвергнуть гипотезу о наличии единичного корня**, то есть ряд считается нестационарным.
  На уровне 10% (p-value < 0.1, критическое значение -2.568) стационарность уже могла бы быть принята, но для ARIMA обычно используют более строгий уровень 5%.

- После применения первой разности ADF‑статистика резко падает до -10.714, p-value ≈ 0.000, что намного меньше 0.01.
  Во всех стандартных критических точках (1%, 5%, 10%) гипотеза о единичном корне отвергается.
  Это означает, что разностный ряд можно считать стационарным, и использование порядка интегрирования `d = 1` в SARIMA‑модели обосновано.

In [ ]:
# 6.3 SARIMA-модель с недельной сезонностью (7 дней)

# Разделяем ряд на train/test по времени (например, последние 20% на тест)
train_size = int(len(ts_daily) * 0.8)
train, test = ts_daily.iloc[:train_size], ts_daily.iloc[train_size:]

print(f'Обучающая выборка: {len(train)} дней, тестовая: {len(test)} дней')

# SARIMA(p,d,q)×(P,D,Q, s), где s = 7 (недельная сезонность)
# Параметры выбираем как простую отправную точку: (1,1,1)×(1,0,1,7)
model = sm.tsa.SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

res = model.fit(disp=False)
print(res.summary().tables[1])  # таблица коэффициентов

### 6.3 Интерпретация параметров SARIMA

- Модель SARIMA(1,1,1)×(1,0,1, 7) даёт следующие оценки параметров:
  - AR(1): 0.586 — значимая положительная авторегрессия первого порядка, текущие изменения цены зависят от предыдущих.
  - MA(1): -0.862 — сильный первый порядок скользящего среднего, модель активно использует информацию об ошибках предыдущего шага.
  - Сезонный AR(1) на лаге 7 (неделя): 0.995 — почти единичная сезонная авторегрессия, что указывает на очень сильную недельную цикличность цены:
    значения сильно коррелируют с тем, что было ровно неделю назад.
  - Сезонный MA(1) на лаге 7: -1.000 — дополняет сезонную структуру ошибок, помогая модели подстраиваться под недельный паттерн.

- Все коэффициенты статистически значимы (p-value ≈ 0.000), их доверительные интервалы не включают ноль.
  Это подтверждает наличие как краткосрочной, так и недельной автокорреляции в ряду дневных цен.
- Оценка дисперсии шума `sigma2 ≈ 21.3` отражает остаточную волатильность, которая не объясняется линейной зависимостью от прошлых значений.

In [ ]:
# 6.4 Прогноз SARIMA и оценка качества

pred = res.get_forecast(steps=len(test))
pred_mean = pred.predicted_mean
pred_ci = pred.conf_int()

mae = mean_absolute_error(test, pred_mean)
rmse = np.sqrt(mean_squared_error(test, pred_mean))

print(f'MAE (прогноз SARIMA): {mae:.2f} €/МВт·ч')
print(f'RMSE (прогноз SARIMA): {rmse:.2f} €/МВт·ч')

# Визуализация прогноза vs факта
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train.index, y=train,
    mode='lines',
    name='Train',
    line=dict(color='#7F8C8D', width=1)
))

fig.add_trace(go.Scatter(
    x=test.index, y=test,
    mode='lines',
    name='Test (факт)',
    line=dict(color='#2C3E50', width=2)
))

fig.add_trace(go.Scatter(
    x=pred_mean.index, y=pred_mean,
    mode='lines',
    name='Прогноз SARIMA',
    line=dict(color='#E67E22', width=2)
))

# Доверительный интервал
fig.add_trace(go.Scatter(
    x=pred_ci.index,
    y=pred_ci.iloc[:, 0],
    mode='lines',
    line=dict(color='#E67E22', width=0, dash='dot'),
    showlegend=False
))
fig.add_trace(go.Scatter(
    x=pred_ci.index,
    y=pred_ci.iloc[:, 1],
    mode='lines',
    fill='tonexty',
    line=dict(color='#E67E22', width=0, dash='dot'),
    name='95% ДИ',
    opacity=0.2
))

fig.update_layout(
    title='Прогноз дневной цены электроэнергии (SARIMA)',
    template='plotly_white',
    xaxis_title='Дата',
    yaxis_title='Цена (€/МВт·ч)',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

### 6.4 Качество прогноза SARIMA и выводы

- Для тестового периода (293 дня) модель SARIMA даёт:
  - MAE ≈ 16.74 €/МВт·ч,
  - RMSE ≈ 18.35 €/МВт·ч.

- Эти значения ощутимо выше, чем средняя внутренняя волатильность, и заметно больше MAE линейной регрессии по почасовым данным (~12 €/МВт·ч), несмотря на то, что здесь используется сглаженный дневной ряд.
  Это показывает, что чисто временная модель, использующая только прошлые цены, **не обеспечивает высокой точности прогноза** для испанского энергорынка.

- Визуально прогноз SARIMA довольно хорошо воспроизводит общий уровень и недельную сезонность цены, но:
  - сглаживает экстремальные пики и провалы;
  - запаздывает при резких сменах режима рынка (аномальные периоды высоких или очень низких цен).

- Причина в том, что SARIMA использует только историю `price actual` и не учитывает ключевые внешние факторы, которые мы видели в других разделах:
  - суммарную нагрузку (`total load actual`),
  - долю возобновляемых источников (`renewable_share`),
  - погодные условия (температура, ветер, облачность и т.д.).

- Таким образом, SARIMA в данной работе выполняет роль **базовой временной модели**:
  - подтверждает наличие сильной недельной сезонности и автокорреляции цен;
  - показывает, что одной только информации о прошлых ценах недостаточно для точного прогноза.

- Для улучшения качества прогноза разумным следующим шагом были бы:
  - модели типа ARIMAX/SARIMAX с регрессорами (нагрузка, ВИЭ, погода);
  - или современные ML‑подходы (градиентный бустинг, нейросети), которые могут одновременно использовать временные лаги и внешние факторы.

## 7. Выводы и рекомендации

### 7.1 Основные эмпирические результаты

1. **Сезонность и суточные паттерны цены**
   - Анализ почасовых и дневных данных показал выраженную сезонность цены и нагрузки: зимой и летом наблюдаются устойчивые пики, связанные соответственно с отоплением и кондиционированием.
   - Тепловая карта «час × месяц» выявила типичный для энергорынков профиль с утренним и вечерним пиками (7–10 и 19–22 часов) и минимальными ценами в ночные часы.
   - В целом, цена формируется на пересечении годовой сезонности (зима/лето) и суточной цикличности спроса.

2. **Роль возобновляемой генерации (ВИЭ)**
   - Корреляционный анализ показал устойчивую отрицательную связь между `price actual` и долей возобновляемых источников (`renewable_share`), а также ветровой генерацией (`generation wind onshore`).
   - Сезонные агрегации генерации показали, что весной доля ВИЭ максимальна, доля ископаемых источников минимальна, и это совпадает с более низким уровнем цен.
   - Визуальный анализ временных рядов «ветровая генерация + цена» подтвердил: периоды повышенного ветра сопровождаются заметным снижением цены, особенно зимой.

3. **Структура генерации по сезонам**
   - Весна характеризуется наибольшей долей ВИЭ в среднесуточной генерации и уменьшением роли газовых и угольных станций.
   - Зимой и летом увеличивается вклад ископаемых источников: зимой для покрытия отопительного спроса, летом — из‑за нагрузки на кондиционирование.
   - Ядерная генерация остаётся относительно стабильной во всех сезонах, играя роль базовой нагрузки.

### 7.2 Результаты моделей регрессии и классификации

4. **Регрессия (прогнозирование фактической цены)**
   - Линейные модели (Linear, Ridge, Lasso) на наборе признаков (нагрузка, генерация по источникам, погода, временные признаки) показали MAE порядка 12 €/МВт·ч и отрицательный R² (≈ -0.5).
   - Отрицательный R² означает, что модель объясняет вариацию хуже, чем наивный константный прогноз «всегда средняя цена», несмотря на использование большого числа разумных признаков.
   - Линейная спецификация плохо справляется с высокой нелинейностью, резкими скачками и сложными взаимодействиями факторов на либерализованном энергорынке.

5. **Классификация уровней цены (низкий / средний / высокий)**
   - При дискретизации `price actual` по квантилям (33% и 66%) классы получаются примерно сбалансированными, что формально благоприятно для классификации.
   - Логистическая регрессия даёт невысокое качество: accuracy порядка 0.35 и F1‑меры по классам далеки от 1.
   - Модель хорошо отличает «крайние» режимы (очень низкие и очень высокие цены), но хуже справляется с промежуточным «средним» уровнем, что отражает пересечение факторов спроса, генерации и погоды.

### 7.3 Выводы по PCA

6. **Снижение размерности и интерпретация компонент**
   - Scree plot показал, что первые две компоненты объясняют около 40% дисперсии (≈22% для PC1 и ≈18% для PC2), а первые 5–6 компонент — уже большую часть вариации.
   - По нагрузкам признаков первая компонента (PC1) описывает «дорогой фоссильный режим»: положительные нагрузки у `generation fossil gas`, `generation fossil hard coal`, `total load actual` и `price actual`, отрицательные — у `renewable_share` и ветровой генерации.
   - Вторая компонента (PC2) отражает в основном погодные и суточные эффекты: в отрицательную сторону тянут температура, скорость ветра, солнечная генерация и час суток, в положительную — влажность.

7. **Распределение сезонов в пространстве PC1–PC2**
   - Точки, соответствующие зиме и лету, смещены вправо по PC1, ближе к области высокой нагрузки, фоссильной генерации и более высоких цен.
   - Весенние наблюдения тяготеют к левой части PC1, что соответствует более высокой доле ВИЭ и относительно более низкому ценовому уровню.
   - Таким образом, PCA даёт компактное двумерное представление, в котором естественным образом разделяются сезонные режимы рынка: «зима/лето с высоким спросом и фоссильной генерацией» и «весна/часть осени с более сильной ролью ВИЭ».

### 7.4 Временные ряды и ARIMA

8. **Стационарность и сезонность**
   - Дневной ряд `price actual` оказывается погранично нестационарным: ADF‑статистика -2.798 при p-value ≈ 0.059 не позволяет уверенно отвергнуть гипотезу о единичном корне на уровне 5%.
   - После первой разности ряд становится явно стационарным (ADF ≈ -10.7, p-value ≈ 0), что оправдывает использование порядка интегрирования d = 1 в SARIMA.
   - Оценённая модель SARIMA(1,1,1)×(1,0,1,7) показала очень сильную недельную сезонную автокорреляцию: коэффициент AR(1) по недельному лагу близок к 1, что отражает устойчивый недельный паттерн цен.

9. **Качество прогноза SARIMA**
   - На тестовом периоде (293 дня) SARIMA даёт MAE ≈ 16.7 €/МВт·ч и RMSE ≈ 18.4 €/МВт·ч, что выше, чем ошибка линейной регрессии по почасовым данным.
   - Модель reasonably хорошо воспроизводит общий уровень и недельную сезонность, но сглаживает экстремальные пики и провалы и запаздывает при резких сменах ценового режима.
   - Это подтверждает, что чисто временная модель, использующая только прошлые значения цены, не способна в одиночку объяснить высокую волатильность и экстремальные события на энергорынке.

### 7.5 Ответ на исследовательский вопрос и рекомендации

10. **Ответ на главный вопрос работы**
    - Анализ EDA, регрессии, PCA и ARIMA согласованно показывает:
      - Сезонность (зима/лето) и суточные паттерны спроса напрямую связаны с повышением уровня цен.
      - Увеличение доли возобновляемой генерации, особенно ветровой, приводит к снижению рыночной цены: это видно по отрицательным корреляциям, структуре PCA (противоположные нагрузки `price actual` и `renewable_share`) и временным графикам.
    - Таким образом, **сезонность и доля ВИЭ существенно влияют на цену электроэнергии в Испании**: высокая доля ВИЭ смягчает ценовое давление, тогда как периоды высокого спроса при недостатке ВИЭ сопровождаются ростом цен.

11. **Практические рекомендации и направления развития модели**
    - Для операционных задач прогнозирования цен простые линейные модели и базовая SARIMA недостаточны: они либо игнорируют временную структуру, либо внешние факторы.
    - Перспективным направлением являются:
      - расширенные временные модели (SARIMAX/ARIMAX) с регрессорами: нагрузка, ВИЭ, погодные показатели;
      - модели машинного обучения (градиентный бустинг, Random Forest, нейросети), обученные на лаговых признаках цены и факторов генерации/погоды;
      - отдельное моделирование экстремальных событий (пиков и отрицательных цен), где линейные предположения явно не работают.
    - С точки зрения энергетической политики результаты подтверждают, что **расширение доли возобновляемой генерации** (особенно ветровой) помогает снижать и стабилизировать цены на оптовом рынке электроэнергии, однако для предотвращения экстремальных скачков нужны также инструменты управления спросом, гибкая генерация и развитие систем накопления энергии.

## 8. Библиография

1. ENTSO-E Transparency Platform — исходные данные по генерации, нагрузке и ценам на электроэнергию в Испании (через агрегированный датасет Kaggle).
2. OpenWeatherMap API — метеорологические данные (температура, влажность, ветер, осадки) для городов Испании, использованные в датасете `weather_features`.
3. Kaggle: “Electricity Consumption & Weather in Spain” — объединённый набор данных по энергосистеме Испании (energy_dataset.csv, weather_features.csv).
4. James G., Witten D., Hastie T., Tibshirani R. *An Introduction to Statistical Learning with Applications in R* — главы по линейной регрессии, классификации и оценке моделей.
5. Bishop C. *Pattern Recognition and Machine Learning* — разделы по линейным моделям и методам снижения размерности (PCA).
6. Box G., Jenkins G., Reinsel G., Ljung G. *Time Series Analysis: Forecasting and Control* — классический источник по моделям ARIMA/SARIMA.
7. Документация `scikit-learn` — реализация регрессии, классификации и PCA: https://scikit-learn.org
8. Документация `statsmodels` — реализация ARIMA/SARIMA и ADF‑теста: https://www.statsmodels.org
9. Документация Plotly — интерактивные визуализации: https://plotly.com/python

## 9. Вклад участников

- **Студент 1**
  - Формулировка темы и исследовательского вопроса.
  - Загрузка и очистка данных (`energy_dataset`, `weather_features`), обработка пропусков и выбросов.
  - Разведочный анализ данных (EDA) и интерактивные визуализации (распределения, сезонность, тепловые карты, структура генерации).

- **Студент 2**
  - Конструирование признаков (доля ВИЭ, временные признаки, погодные агрегаты).
  - Построение и оценка регрессионных моделей (Linear, Ridge, Lasso) для `price actual`.
  - Построение и оценка моделей классификации уровней цены (логистическая регрессия, анализ матрицы ошибок и метрик).

- **Студент 3**
  - Применение PCA: выбор признаков, обучение, анализ объяснённой дисперсии, интерпретация компонент и biplot.
  - Анализ временных рядов: построение дневного ряда `price actual`, ADF‑тест, настройка и оценка SARIMA‑модели.
  - Сводные выводы и формулировка рекомендаций по развитию моделей и энергетической политике.

## 10. Приложения

В рамках данной работы все вычисления, построение моделей и визуализаций реализованы в Jupyter Notebook.

К приложением относятся:
- Полный код предобработки данных (очистка, генерация признаков, объединение `energy_dataset` и `weather_features`).
- Код и дополнительные графики EDA, не вошедшие в основной текст (альтернативные масштабы, разрезы по годам и городам).
- Полный вывод моделей регрессии, классификации, PCA и SARIMA (таблицы коэффициентов, дополнительные метрики).
- Дополнительные диагностические графики для временных рядов (ACF/PACF, остатки моделей), доступные в ноутбуке для при необходимости более детального анализа.